In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "holidays", "pytz", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', 'holidays', 'pytz', '-q'], returncode=0)

In [2]:
import os
import sys
from pyspark.sql import SparkSession

ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")
CONN_STR     = os.environ["AZURE_STORAGE_CONNECTION_STRING"]

spark = (SparkSession.builder
    .appName("Bronze_to_Silver")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages", 
            "org.apache.hadoop:hadoop-azure:3.3.4,"
            "io.delta:delta-spark_2.13:4.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Driver Python: {sys.executable}")

Spark version: 4.0.0
Master: spark://spark-master:7077
Driver Python: /opt/conda/bin/python


In [4]:
import io
import pandas as pd
from azure.storage.blob import BlobServiceClient

blob_service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = blob_service.get_container_client(CONTAINER)

# 1. List file
print("FILES TRONG gold/predictions/")
blobs = list(container_client.list_blobs(name_starts_with="gold/predictions/"))
for b in blobs:
    print(f"  {b.name}  ({b.size} bytes)")
print(f"Tổng: {len(blobs)} file\n")

# 2. Đọc tất cả parquet gộp lại
dfs = []
for b in blobs:
    if not b.name.endswith(".parquet"):
        continue
    data = blob_service.get_blob_client(
        container=CONTAINER, blob=b.name
    ).download_blob().readall()
    dfs.append(pd.read_parquet(io.BytesIO(data)))

if not dfs:
    print("Không có file parquet nào trong gold/predictions/")
else:
    df = pd.concat(dfs, ignore_index=True)
    print("SCHEMA")
    print(df.dtypes)
    print(f"\nTỔNG QUAN")
    print(f"Tổng rows        : {len(df)}")
    print(f"Unique link_id   : {df['link_id'].nunique()}")
    print(f"Timestamp range  : {df['timestamp'].min()}  →  {df['timestamp'].max()}")
    print(f"\nPHÂN PHỐI predicted_congestion")
    vc = df['predicted_congestion'].value_counts().sort_index()
    vc.index = vc.index.map({0:'0-Thông thoáng', 1:'1-Tắc nghẽn', 2:'2-Nghiêm trọng'})
    print(vc)
    print(f"\nCONFIDENCE")
    print(df['confidence'].describe().round(4))
    print(f"\nNULL CHECK")
    print(df.isnull().sum())
    print(f"\n10 DÒNG MỚI NHẤT")
    print(df.sort_values("timestamp", ascending=False).head(10).to_string(index=False))

FILES TRONG gold/predictions/
  gold/predictions/date=2026-06-30/hour=13/part-1782841153.parquet  (8510 bytes)
  gold/predictions/date=2026-06-30/hour=14/part-1782844085.parquet  (8399 bytes)
  gold/predictions/date=2026-06-30/hour=14/part-1782844406.parquet  (8424 bytes)
  gold/predictions/date=2026-06-30/hour=14/part-1782844702.parquet  (8472 bytes)
  gold/predictions/date=2026-06-30/hour=14/part-1782844961.parquet  (8433 bytes)
  gold/predictions/date=2026-06-30/hour=14/part-1782845275.parquet  (8463 bytes)
Tổng: 6 file

SCHEMA
link_id                         object
borough                         object
link_name                       object
timestamp               datetime64[ns]
target_time             datetime64[ns]
predicted_congestion             int64
confidence                     float64
dtype: object

TỔNG QUAN
Tổng rows        : 555
Unique link_id   : 94
Timestamp range  : 2026-06-30 13:34:00  →  2026-06-30 14:46:02

PHÂN PHỐI predicted_congestion
predicted_congestion
0-Th